# Freemarket Medallion Pipeline — orchestrator

This notebook **orchestrates and narrates**; all transformation logic lives in the tested
`src/` package (see `plan/SPEC.md` § Engineering Standards). Run top-to-bottom
(Restart & Run All) from an empty warehouse.

Layers built so far: **Bronze foundation** (six schemas) → **Silver `core`** (companies unpick).

In [1]:
# Make the repo root importable when running from notebook/
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config, warehouse, silver_core
from src.pipeline import build_foundation
config.WAREHOUSE_PATH

PosixPath('/Users/hallabehery/Documents/My Projects/FM-Data-Engineer-Test-HS/submission/warehouse.duckdb')

## 1. Foundation — six schemas
Connect (creates the file if absent) and create the schemas. Idempotent.

In [2]:
con = warehouse.connect()
build_foundation(con)

18:16:04 | INFO  | [foundation] schemas ready: raw, live, core, shape, data_mart, curated


('raw', 'live', 'core', 'shape', 'data_mart', 'curated')

## 2. Silver `core` — companies unpick
Flatten `companies.json` into `core.companies` (one row per `dcId`), exposing nested
registration / classification / financials / footprint fields and the `parent_group_id`
bridge key. The stage report shows row-count conservation.

In [3]:
report = silver_core.build_companies(con)
print(report)

18:16:04 | INFO  | [silver.core.companies] in=44 out=44 kept=44 quarantined=0 dropped=0 conserved=yes


[silver.core.companies] in=44 out=44 kept=44 quarantined=0 dropped=0 conserved=yes


In [4]:
con.sql("""
    SELECT dc_id, legal_name, vertical, financials_currency,
           parent_group_id, parent_group_role
    FROM core.companies
    ORDER BY dc_id
    LIMIT 10
""").df()

,dc_id,legal_name,vertical,financials_currency,parent_group_id,parent_group_role
0,234850008297,Payment One Holding S.à r.l.,Holding,USD,167898894567,HOLDING
1,234854669542,Remitance Central Holdings Ltd,Holding,USD,167986897142,HOLDING
2,234939159769,Payment One Operating Company Ltd,MSB,USD,167898894567,OPERATING
3,234939159775,Remitance Central (Europe) Ltd,MSB,USD,167986897142,OPERATING
4,258599414982,Get Tranzzact (Europe) Ltd,MSB,USD,187752508616,OPERATING
5,258740050135,ConstructGame Technology Pvt Ltd,Technology,USD,187754310858,TECHNOLOGY
6,258740050139,Sanno Payments Operations Ltd,MSB,USD,187757905092,OPERATING
7,258863167682,ConstructGame Group Holdings Ltd,Holding,USD,187754310858,HOLDING
8,258929414331,ConstructGame Media Ltd,Marketing,USD,187754310858,MARKETING
9,258962745548,ConstructGame Group Holdings Ltd,Holding,USD,187754310858,HOLDING


## Inspect the built warehouse
Reopen `submission/warehouse.duckdb` **read-only** to see the schemas and the tables built
so far — this is how you'd open the warehouse file from anywhere with the `duckdb` package.

In [5]:
con.close()
import duckdb
inspect = duckdb.connect(str(config.WAREHOUSE_PATH), read_only=True)
tables = inspect.sql("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    ORDER BY table_schema, table_name
""").df()
inspect.close()
tables

,table_schema,table_name
0,core,companies
